# Mathias Henry Morgan

## 35087056

## Assessment 2: Big data project - Phase 3

## Machine Learning Pipeline: XGBoost Customer Churn Prediction

## Instructions

1. ***Run all*** to import libraries, define function logic and trigger the execute_ml_pipeline() function.
2. The cell block containing execute_ml_pipeline() will contain logging statements for visibility of actions. performed
3. At the completion of the run a cleaned file will be generated with a unique date time stamp.

*** Note *** 
The script may take a few minutes run if running on local enviroment.

## Model Evaluation Metrics

The following metrics are used to evaluate the performance of the XGBoost Base and Optimised churn prediction models:

- **F1 Score**  
  The harmonic mean of precision and recall. It balances the trade-off between false positives and false negatives, and is especially useful when class distributions are imbalanced. A higher F1 score indicates stronger overall performance.

- **Precision**  
  The proportion of true positive predictions out of all positive predictions made by the model.

- **Recall**  
  The proportion of true positive predictions out of all actual positive cases in the data.

- **Accuracy**  
  The overall percentage of correct predictions made by the model. While commonly used, it may be misleading for imbalanced datasets.

- **AUC-ROC (Area Under the Receiver Operating Characteristic Curve)**  
  Measures the model's ability to distinguish between classes across all threshold values. A higher AUC indicates better class separation:  
  - An AUC of **1.0** represents perfect classification.  
  - An AUC of **0.5** suggests performance no better than random guessing.  

  Important metric in classification problems, as it evaluates how well the model ranks positive instances relative to negative ones regardless of decision threshold.

# Libraries

In [65]:
!pip install xgboost==1.7.6
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.ml.stat import Correlation
from pyspark.sql.functions import col, abs
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier, GBTClassificationModel, ClassificationModel
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import when
from pyspark.sql import Row
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import lit
from pathlib import Path
from pyspark import StorageLevel
from xgboost.spark import SparkXGBClassifier
import uuid
import warnings
import pandas as pd
import time
from datetime import timedelta
warnings.filterwarnings('ignore')

## Spark Configurations

This code block initializes a local Spark session and configures parallel processing settings to optimize performance. Specifically, it:

- Launches a Spark application in local mode, utilizing all available CPU cores on the machine.
- Determines how many tasks Spark can run in parallel based on the available cores.
- Dynamically sets the number of **shuffle partitions**, which are used during wide transformations.

Shuffle operations can be computationally expensive because they involve redistributing data across the cluster (or cores, in local mode). By adjusting the number of shuffle partitions to align with the hardware's parallel processing capacity, we can reduce execution time and avoid common performance bottlenecks like task skew or underutilization of CPU resources. Tuning this configuration helps ensure that Spark workloads are more efficient and scalable.


In [66]:
spark = SparkSession.builder.appName("ChurnXGBoost").master("local[*]").getOrCreate() # create spark session
cores = spark.sparkContext.defaultParallelism # sets default
target_shuffle_partitions = cores * 2 # set custom partitions for shuffling based on available cores
spark.conf.set("spark.sql.shuffle.partitions", target_shuffle_partitions) # will use above coun during shuffle operations
print("spark.sql.shuffle.partitions ->", spark.conf.get("spark.sql.shuffle.partitions")) # displays shuffle partitions

spark.sql.shuffle.partitions -> 24


## Concatenate Cleaned Churn Files

This function loads and combines multiple cleaned churn CSV files (`churn_clean*.csv`) from the current directory into a single Spark DataFrame. It concatenates all available files into one unified dataset, enabling the modelling of all available data at once. The primary purpose is to automate the ingestion of one or more cleaned files while ensuring schema consistency across them, making the process scalable and error-resistant.

In [67]:
def concatenate_clean_churn_files() -> DataFrame | None:
    """
    Combines all 'churn_clean*.csv' files in the current directory into a single Spark DataFrame.
    The first matching file determines the schema. All subsequent files are appended.
    If no matching files are found, the function returns None.
    Returns:
        churnCombinedDf: Spark DataFrame containing the combined data, or None if no files are found.
    """
    csvFiles = [str(file) for file in Path(".").glob("churn_clean*.csv")] # find all files with chun_clean name
    print(f"All available cleaned churn csv files: {csvFiles}")
    if not csvFiles:
        print("No churn_cleaned CSV files found...")
        return None
    count = 0 # set loop count
    for fileName in csvFiles:
        if count == 0:
            churnCombinedDf = spark.read.csv(fileName, header=True, inferSchema=True) # create initial spark df
            count += 1
        else:
            newChurnDf = spark.read.csv(fileName, header=True, inferSchema=True)
            churnCombinedDf = churnCombinedDf.unionByName(newChurnDf) # concatanate secondary spark df to the previous
    return churnCombinedDf

## Display Metadata

This function provides a quick overview of the churn dataset by displaying basic metadata about the input Spark DataFrame. Specifically, it:

- Converts the Spark DataFrame to a Pandas DataFrame for cleaner and more detailed output.
- Prints column-level metadata including data types, non-null value counts, and memory usage.

The purpose of this function is to help the user verify the structure and completeness of the dataset before it is used for downstream processing and modeling. By surfacing potential issues such as missing values or incorrect data types early, it supports better data quality control and helps prevent errors during later stages of the pipeline.

In [68]:
def get_churn_metadata(churnDataFrame: DataFrame) -> None:
    """
    Displays basic metadata about the churn DataFrame, including data types and non-null counts.
    Converts the Spark DataFrame to a Pandas DataFrame for cleaner output.
    Args:
        churnDataFrame: Spark DataFrame containing the churn dataset.
    Returns:
        None. Prints metadata to the console.
    """
    tempPandasDf = churnDataFrame.toPandas()
    tempPandasDf.info() # print metadata info

## Data Splitting by Label

This function prepares the churn dataset for modeling by splitting it into labeled and unlabeled subsets based on the presence of the `churn_risk_score` column. Specifically, it:

- Filters rows where `churn_risk_score` is present into the **labeled dataset**.
- Filters rows where `churn_risk_score` is missing into the **unlabeled dataset**.
- If no unlabeled data exists, reserves 1% of the labeled data for demonstration purposes to simulate prediction.

This logic ensures the pipeline can always support a prediction phase, even when the input data is fully labeled. In such cases, a small portion is held back (1%) and restructured for use in testing model predictions. This includes renaming `churn_risk_score` to `actual_churn_risk_score` and adding an empty `predicted_churn_risk_score` column for the reserved set.

By automating this data split, the pipeline remains flexible and robust across both training and evaluation workflows, whether in production or demo scenarios.

In [69]:
def split_dataframe_by_label(churnDataFrame: DataFrame) -> tuple[DataFrame, DataFrame]:
    """
    Splits DataFrame into labeled and unlabeled sets based on if churn_risk_score contains empty rows.
    If no unlabeled data exists, reserves 1% of labeled data for demonstrative prediction purposes.
    Args:
        churnDataFrame: Input DataFrame containing cleaned churn data.
    Returns:
        churnlabeled: Rows with churn_risk_score.
        churnunlabeled: Rows without churn_risk_score or reserved data.
        reservedFlag: Boolean True if data was reserved.
    """
    # split data frames by churn_risk value label
    churnlabeled = churnDataFrame.filter(col('churn_risk_score').isNotNull())
    churnunlabeled = churnDataFrame.filter(col('churn_risk_score').isNull())
    reserve = 0.01 # default to 1% reserve rate if no churn_risk present
    reservedFlag = False
    if churnunlabeled.count() == 0:
        reservedFlag = True
        print('no empty churn_risk_score rows...')
        print(f"Reserving {int(1)}% of labeled data for demo predictions...")
        trainDf, reservedDf = churnlabeled.randomSplit([1 - reserve, reserve], seed=42) # apply 1% split
        churnlabeled = trainDf
        churnunlabeled = reservedDf
        print(f"Reserving dataframe with {churnunlabeled.count()} rows...")
        churnunlabeled = churnunlabeled.withColumnRenamed("churn_risk_score", "actual_churn_risk_score")
        churnunlabeled = churnunlabeled.withColumn("predicted_churn_risk_score", lit(None))
    return churnlabeled, churnunlabeled

## Separate Features and Target

This function prepares the churn dataset for modeling by separating the input DataFrame into feature columns and a target column. Specifically, it:

- Selects all columns **except the last one** as input features.
- Treats the **last column** as the target variable to be predicted (churn risk score is always the last column).
- Returns both the list of feature column names and the target column name.

Automating this step ensures consistency in how input data is structured and simplifies integration with feature engineering and modeling stages of the pipeline.

In [70]:
def separate_feature_target_cols(churnDataFrame: DataFrame) -> tuple[list,str]:
    """
    Separates the input DataFrame columns into features and target variable.
    Uses all columns except the last as features, and the last column as target.
    Args:
        churnDataFrame: Input DataFrame containing cleaned churn data.
    Returns:
        featureCols: List of all feature column names.
        targetCol: Name of the target column to predict.
    """
    featureCols = churnDataFrame.columns[:-1] # seperate feature columns
    targetCol = churnDataFrame.columns[-1] # seperate target column
    print(f"All Feature Columns: {featureCols}")
    print(f"Target Column: {targetCol}")
    return featureCols, targetCol

## Correlation Analysis

This function performs Pearson correlation analysis between each feature and the target variable to identify their predictive strength. Specifically, it:

- Calculates the correlation coefficient between each feature and the target.
- Categorizes the strength of each feature as **strong**, **moderate**, or **weak** based on user-defined threshold values.
- Displays a ranked list of features by absolute correlation strength.
- Returns only those features with strong or moderate correlation to the target.

This step is used downstream for **feature selection**, allowing the pipeline to focus on the most predictive inputs and reduce noise from weak features. 

In [71]:
def correlation_analysis(churnDataFrame: DataFrame, featureCols: list,
                         targetCol: str, moderateVal: float, strongVal: float) -> list:
    """
    Performs correlation analysis between features and target variable.
    Identifies features with strong, moderate and weak correlation based on threshold values.
    Args:
        churnDataFrame: Input DataFrame containing features and target.
        featureCols: List of feature column names to analyze.
        targetCol: Name of the target column.
        moderateVal: Threshold for moderate correlation.
        strongVal: Threshold for strong correlation.
    Returns:
        list: Features showing strong/moderate correlation with target.
    """
    print("Running correlation analysis...")

    corrVals = [] # create list to store correlation values
    for feature in featureCols:
        corr = churnDataFrame.stat.corr(feature, targetCol)
        corrVals.append((feature, corr))

    # Create data frame of correaltion values sorted highest to lowes
    corrDf = spark.createDataFrame(corrVals, ["feature", "pearson_correlation"]) \
                  .orderBy(abs(col("pearson_correlation")).desc())
    corrDf = corrDf.withColumn("feature_strength", # distinguish between, strong, moderate and weak values
         when(abs(col("pearson_correlation")) > strongVal, "Strong")
        .when(abs(col("pearson_correlation")) > moderateVal, "Moderate")
        .otherwise("Weak"))

    print("All Correlation Scores:")
    corrDf.show(n=corrDf.count(), truncate=False) # display correaltion results
    print("Features displaying the strongest predictive signal:")
    filteredCorrDf = corrDf.filter(col("feature_strength").isin("Strong", "Moderate"))
    filteredCorrDf.orderBy(abs(col("pearson_correlation")).desc()).show(truncate=False)
    strongCorrFeatures = filteredCorrDf.select("feature").rdd.flatMap(lambda x: x).collect() # create RDD to get list of strong features
    return strongCorrFeatures

## Feature Assembly and Data Splitting

This function prepares the churn dataset for machine learning by assembling features into a single vector and splitting the data into training and test sets. Specifically, it:

- Uses `VectorAssembler` to combine all selected feature columns into a single `"features"` vector column required by Spark ML and XGBoost.
- Wraps the transformation in a `Pipeline` to ensure consistency and reusability.
- Performs a random train-test split based on user-specified proportions.

This step is essential for converting structured tabular data into a format suitable for model training. It ensures that the input features are properly vectorized and that the model has clearly defined training and evaluation datasets.

In [72]:
def feature_assemble(churnDataframe, featureCols,
                     trainSplit:float, testSplit:float) -> tuple[DataFrame, DataFrame, bool]:
    """
    Combines features into vector format and splits data into training and test sets.
    Creates feature vectors for XGBoost and performs train-test split as specified.
    Args:
        churnDataframe: Input DataFrame containing features
        featureCols: List of feature column names to assemble
        trainSplit: Proportion of data for training (%)
        testSplit: Proportion of data for testing (%)
    Returns:
        trainDf: Training dataset with feature vectors
        testDf: Test dataset with feature vectors
    """
    print("Assembling features...")
    print("Combining all features into single vector...")
    assembleAll = VectorAssembler(inputCols = featureCols,
                                  outputCol="features") # assemble features into one vector
    pipeline = Pipeline(stages=[assembleAll])
    vectorDf = pipeline.fit(churnDataframe).transform(churnDataframe) # fit pipeline to data
    print("Splitting data...")
    print(f"{trainSplit*100}/{testSplit*100} Test Train Split...")
    trainDf, testDf = vectorDf.randomSplit([trainSplit, testSplit], seed=42) # train, test split
    return trainDf, testDf

## Base Model Training and Evaluation

This function trains a base XGBoost model using all available features and evaluates its performance on a test set. Specifically, it:

- Creates an instance of `SparkXGBClassifier` using the vectorized feature column and the specified target.
- Trains the model on the training DataFrame and generates predictions on the test DataFrame.
- Evaluates the model using standard classification metrics.
- Constructs a confusion matrix.

The purpose of this function is to establish a benchmark model against which future tuning and optimization efforts can be compared. By reporting key metrics and displaying the confusion matrix, it provides a comprehensive view of the model’s initial predictive performance.

In [73]:
def baseline_model(trainDf, testDf, targetCol) -> tuple[DataFrame, str]:
    """
    Trains and evaluates a baseline XGBoost model using all available features.
    Returns the trained model and its F1 score on the test set.
    Args:
        trainDf: Training DataFrame containing feature vectors
        testDf: Test DataFrame for model evaluation
        targetCol: Name of the target/label column
    Returns:
        xgbModel: Trained XGBoost model
        baselineF1Score: F1 score achieved on test data
    """
    print("Training base model...")
    xgbClassifier = SparkXGBClassifier(features_col="features", label_col=targetCol,
                                       prediction_col="prediction", num_workers=cores) # Create XGB Classifier model
    xgbModel = xgbClassifier.fit(trainDf) # apply model to train data set
    predictions = xgbModel.transform(testDf) # apply to make predictions

    # metrics to analyse model performance - F1, Precision, Recall, Accuracy, AUC ROC
    evaluator = MulticlassClassificationEvaluator(labelCol=targetCol,predictionCol="prediction",metricName="f1")
    baseF1Score = round(evaluator.evaluate(predictions),5)
    precision = evaluator.setMetricName("weightedPrecision").evaluate(predictions)
    recall = evaluator.setMetricName("weightedRecall").evaluate(predictions)
    acc = MulticlassClassificationEvaluator(labelCol=targetCol,predictionCol="prediction",metricName="accuracy").evaluate(predictions)
    auc = BinaryClassificationEvaluator(labelCol=targetCol,rawPredictionCol="rawPrediction", metricName="areaUnderROC").evaluate(predictions)

    # create and visualise confusion matrix
    confusionMatrix = predictions.groupBy("prediction", "churn_risk_score").count()
    confusionMatrix = confusionMatrix.withColumn("prediction", confusionMatrix["prediction"].cast(IntegerType()))
    confusionMatrix = confusionMatrix.withColumn("metric", # add metrics fields based on integer values
     when((col("prediction") == 1) & (col("churn_risk_score") == 1), "TP")
    .when((col("prediction") == 1) & (col("churn_risk_score") == 0), "FP")
    .when((col("prediction") == 0) & (col("churn_risk_score") == 1), "FN")
    .when((col("prediction") == 0) & (col("churn_risk_score") == 0), "TN"))
    
    # display metrics
    print(f"Base Model F1 Score: {round(baseF1Score, 5)}")
    print(f"Base Model Precision: {round(precision, 5)}")
    print(f"Base Model Recall: {round(recall, 5)}")
    print(f"Base Model Accuracy: {round(acc, 5)}")
    print(f"Base Model ROC AUC: {round(auc, 5)}")
    print("Base Model Confusion Matrix:")
    confusionMatrix.show()
    return xgbModel, baseF1Score

## Feature Importance by Information Gain

This function analyses feature importance scores based on **information gain** extracted from a trained XGBoost model. Specifically, it:

- Retrieves gain-based importance values for each feature from the XGBoost booster in order.
- Ranks features according to their contribution to model splits, using information gain as the metric.
- Categorises each feature as **Strong**, **Moderate**, or **Weak** based on predefined gain thresholds.
- Displays all features and highlights those with the greatest predictive influence.

This step helps identify which features the model finds most informative. It supports **feature selection** and provides insight into the relative importance of input variables. The function returns a list of features with strong or moderate information gain for use in downstream modelling optimisation.

In [74]:
def feature_information_gain(model, featureCols, strongVal, moderateVal) -> list:
    """
    Extracts and displays feature importances based on information gain from a trained XGBoost model.
    Returns a list of features with strong or moderate importance.
    Args:
        model: Trained XGBoost model from SparkXGBClassifier
        featureCols: List of original feature column names used for training
    Returns:
        strongGainFeatures: List of feature names with 'Strong' or 'Moderate' information gain
    """
    print("Running information gain analysis...")
    importances = model.get_booster().get_score(importance_type="gain") # Extract information gain for features
    print("All features information gain:")
    featureMap = {f"f{i}": name for i, name in enumerate(featureCols)}
    sortedImportances = sorted(importances.items(), key=lambda x: x[1], reverse=True)

    featureDict = {} # create dictionary object to store feature names and scores
    for feature, score in sortedImportances:
        feature = featureMap.get(feature, feature)
        featureDict[feature] = score # set name and score
    scoreList = [[featureMap.get(feat, feat), score] for feat, score in sortedImportances]
    allFeatureGainDf = spark.createDataFrame(scoreList, ["feature", "gain"])
    allFeatureGainDf = allFeatureGainDf.withColumn("gain_strength",
         when(abs(col("gain")) > strongVal, "Strong")
        .when(abs(col("gain")) > moderateVal, "Moderate")
        .otherwise("Weak")) # define strong, moderate and weak gain
    allFeatureGainDf.show(n=allFeatureGainDf.count(), truncate=False)
    print("Features with stronger information gain scores...")
    strongGainDf = allFeatureGainDf.filter(col("gain_strength").isin("Strong", "Moderate")) # filter by strong and moderate
    strongGainDf.orderBy(abs(col("gain_strength")).desc()).show(truncate=False)
    strongGainFeatures = strongGainDf.select("feature").rdd.flatMap(lambda x: x).collect() # create rdd to extract list of strong features
    return strongGainFeatures

## Combine Strong Features

This function unifies the results of two feature selection methods—**correlation analysis** and **information gain** into a single list of strong predictors. Specifically, it:

- Merges the two input lists of selected features.
- Removes duplicates to produce a unique set of high-value features.
- Returns the combined list as a foundation for refined model training.

By integrating statistical and model-based feature selection approaches, this step ensures that the strongest predictors from both perspectives are retained. It supports more focused and efficient downstream modelling.

In [75]:
def get_strong_features(corrFeatures: list, gainFeatures: list) -> list:
    """
    Combines features selected via correlation and information gain into a unified list of strong features.
    Args:
        corrFeatures: List of features selected based on correlation analysis
        gainFeatures: List of features selected based on information gain from the model
    Returns:
        strongFeatures: Combined list of unique features deemed strong by either method
    """
    strongFeatures = list(set(corrFeatures + gainFeatures)) # get set of strong information gain and correaltion features
    print(f"Strong feature list: {strongFeatures}")
    return strongFeatures

## K-Fold Cross-Validation with XGBoost

This function performs k-fold cross-validation using XGBoost classifier and evaluates the optimised model on the test set. Specifically, it:

- Transforms the input features into a vector format using `VectorAssembler`.
- Defines a grid of hyperparameters for tuning (`max_depth`, `learning_rate`, `subsample`).
- Runs k-fold cross-validation on the training data using Spark’s `CrossValidator`.
- Selects the best-performing model based on F1 score.
- Evaluates the selected model on the test set using multiple metrics.
- Builds and displays a labelled confusion matrix for interpretability.

This function is key to improving model performance through automated hyperparameter tuning while ensuring robust validation. It returns the best model from cross-validation along with its F1 score on the test set, providing a benchmark for comparison with the base model.

In [76]:
def kfold_cross_validation(trainDf, testDf, featureCols,
                           targetCol, k=3)-> tuple[GBTClassificationModel, float]:
    """
    Performs k-fold cross-validation using a Gradient-Boosted Tree (GBT) model and evaluates performance on test data.
    Args:
        trainDf: Training DataFrame containing feature columns and target
        testDf: Test DataFrame for final model evaluation
        featureCols: List of selected feature column names
        TargetCol: Name of the target/label column
        modelType: Type of model to train (currently supports "GBT" only)
        k: Number of folds for cross-validation
    Returns:
        bestModel: Best-performing GBT model from cross-validation
        optimisedF1Score: F1 score of the best model evaluated on the test set
    """
    print("Training optimised model...")
    print(f"Running {k}-fold CV on training data...")
    assembler = VectorAssembler(inputCols= featureCols, outputCol="strongFeatures")
    train = assembler.transform(trainDf).select("strongFeatures", targetCol)
    test = assembler.transform(testDf).select("strongFeatures", targetCol)

    # create XGB classifer model
    model = SparkXGBClassifier(label_col=targetCol,
                               features_col="strongFeatures",
                               prediction_col="prediction",
                               num_workers=cores)
    print("Tuning hyperparameters: max_depth, learning_rate, subsample...")
    paramGrid = (ParamGridBuilder() # establish hyperparameter grid
        .addGrid(model.getParam("max_depth"), [3, 5]) \
        .addGrid(model.getParam("learning_rate"), [0.05, 0.1]) \
        .addGrid(model.getParam("subsample"), [0.8, 1.0]) \
        .build())
    evaluator = MulticlassClassificationEvaluator(labelCol=targetCol, predictionCol="prediction", metricName="f1")

    # apply cross validation
    print("Applying cross validation to find best F1 Score...")
    cv = CrossValidator(estimator=model, estimatorParamMaps=paramGrid,evaluator=evaluator,
                        numFolds=k, parallelism=cores)
    cvModel = cv.fit(train)
    predictions = cvModel.transform(test)

    # metrics to analyse model performance - F1, Precision, Recall, Accuracy, AUC ROC
    optimisedF1Score = round(evaluator.evaluate(predictions),5)
    precision = evaluator.setMetricName("weightedPrecision").evaluate(predictions)
    recall = evaluator.setMetricName("weightedRecall").evaluate(predictions)
    acc = MulticlassClassificationEvaluator(labelCol=targetCol,predictionCol="prediction",metricName="accuracy").evaluate(predictions)
    auc = BinaryClassificationEvaluator(labelCol=targetCol,rawPredictionCol="rawPrediction", metricName="areaUnderROC").evaluate(predictions)

    # create and visualise confusion matrix
    confusionMatrix = predictions.groupBy("prediction", "churn_risk_score").count()
    confusionMatrix = confusionMatrix.withColumn("prediction", confusionMatrix["prediction"].cast(IntegerType()))
    confusionMatrix = confusionMatrix.withColumn("metric", # add metrics fields based on integer values
     when((col("prediction") == 1) & (col("churn_risk_score") == 1), "TP")
    .when((col("prediction") == 1) & (col("churn_risk_score") == 0), "FP")
    .when((col("prediction") == 0) & (col("churn_risk_score") == 1), "FN")
    .when((col("prediction") == 0) & (col("churn_risk_score") == 0), "TN"))
    
    # display metrics
    print(f"Optimised Model F1 Score: {round(optimisedF1Score, 5)}")
    print(f"Optimised Model Precision: {round(precision, 5)}")
    print(f"Optimised Model Recall: {round(recall, 5)}")
    print(f"Optimised Model Accuracy: {round(acc, 5)}")
    print(f"Optimised Model ROC AUC: {round(auc, 5)}")
    print("Optimised Model Confusion Matrix:")
    confusionMatrix.show()

    return cvModel.bestModel, optimisedF1Score

## Model Selection

This function compares the F1 scores of the base and optimised models and selects the better-performing one. Specifically, it:

- Compares the F1 scores from both models.
- Selects the model with the higher F1 score.
- Returns the chosen model, the name of the selected model type (`"base"` or `"optimised"`), and the corresponding list of features used.

This step ensures that the final model used in the pipeline reflects the best-performing configuration. It also keeps track of which features were associated with the selected model, supporting reproducibility and downstream reporting. This approach provides flexibility and future-proofs the pipeline by allowing dynamic model selection based on actual performance. It ensures that the most suitable model is used without requiring manual intervention.

In [77]:
def select_best_model(baselineF1, optimisedF1, baseModel,
                      optimisedModel, strongFeatures, allFeatures) -> tuple[ClassificationModel, str, list]:
    """
    Compares baseline and optimised model F1 scores and selects the better-performing model.
    Args:
        baselineF1: F1 score of the baseline model
        optimisedF1: F1 score of the optimised model
        baseModel: Trained baseline model
        optimisedModel: Trained optimised model from cross-validation
        strongFeatures: List of features used in the optimised model
        allFeatures: List of features used in the baseline model
    Returns:
        model: The selected model (either baseline or optimised)
        modelType: String indicating which model was selected ("base" or "optimised")
        features: List of features associated with the selected model
    """
    print(f"Base model F1 Score: {baselineF1}")
    print(f"Optimised model F1 Score: {optimisedF1}")
    if optimisedF1 > baselineF1:
        model = optimisedModel
        modelType = "optimised"
        features = strongFeatures
        print(f"Optimised model selected...")
    else:
        model = baseModel
        modelType = "base"
        features = allFeatures
        print(f"Base model selected...")
    return model, modelType, features

## Predict Churn Risk Score

This function applies the selected model to unlabeled churn data and displays predicted churn risk scores. Specifically, it:

- Assembles the required features into a vector using `VectorAssembler`.
- Applies the trained model (baseline or optimised) to generate predictions.
- If labels (`actual_churn_risk_score` or `churn_risk_score`) are available, it calculates and prints prediction accuracy.
- Displays the predicted vs. actual churn risk scores when applicable.

This function supports both evaluation and demonstration use cases by adapting dynamically to the available data. It provides a consistent prediction interface regardless of the selected model type, and includes logic to report performance when true labels are present.

This design improves flexibility and future-proofs the pipeline, allowing it to handle both labelled and unlabelled scenarios with minimal adjustments.

In [78]:
def predict_churn_score(modelType, model,
                        churnDataUnlabelled: DataFrame, features) -> None:
    """
    Applies the selected model to unlabeled churn data and displays predictions along with accuracy if labels exist.
    Args:
        modelType: Type of model used ("base" or "optimised")
        model: Trained classification model
        churnDataUnlabelled: DataFrame containing data to predict churn scores on
        features: List of feature columns to be used for prediction
    Returns:
        None. Prints prediction results and accuracy (if actual labels are available).
    """
    if modelType == "optimised":
        featureVectorCol = "strongFeatures"
        selectedFeatures = ['actual_churn_risk_score'] + features
    else:
        featureVectorCol = "features"
        selectedFeatures = ['actual_churn_risk_score'] + features

    # list comprehension to select features
    churnDataUnlabelled = churnDataUnlabelled.select(*[feature for feature in selectedFeatures if feature in churnDataUnlabelled.columns])

    # apply model to predictions
    assembler = VectorAssembler(inputCols=features, outputCol=featureVectorCol)
    churnunlabeledPredict = assembler.transform(churnDataUnlabelled)
    predictions = model.transform(churnunlabeledPredict)
    predictions = predictions.withColumnRenamed("prediction", "predicted_churn_risk_score")

    # determine which label column to use
    if "actual_churn_risk_score" in predictions.columns:
        Targetcol = "actual_churn_risk_score"
    elif "churn_risk_score" in predictions.columns:
        Targetcol = "churn_risk_score"

    # if target col is not empty show predictions
    if Targetcol:
        predictionDf = predictions.select(Targetcol, "predicted_churn_risk_score")
        print(f"Prediction results ({modelType} model using '{Targetcol}')")
        predictionDf = predictionDf.withColumn("predicted_churn_risk_score", predictionDf["predicted_churn_risk_score"].cast(IntegerType()))
        predictionDf.show(truncate=False)
        # Find accurate predictions
        correct = predictions.filter(col(Targetcol) == col("predicted_churn_risk_score")).count()
        total = predictions.count()
        accuracy = correct / total if total else 0
        print(f"Prediction Accuracy: {round(accuracy,5)} ({round(correct,5)}/{round(total,5)})")
    else:
        print("No label column found ('actual_churn_risk_score' or 'churn_risk_score') in data.")
        predictions.select("predicted_churn_risk_score").show(truncate=False)

## Execute Machine Learning Pipeline

This function orchestrates the entire end-to-end machine learning pipeline for churn prediction. It automates all key stages of the workflow, including data preparation, feature selection, model training, evaluation, and prediction. Specifically, it:

1. Loads and combines all available cleaned churn datasets.
2. Splits the data into labelled and unlabelled subsets.
3. Trains a **baseline XGBoost model** using all features.
4. Evaluates feature relevance using **correlation analysis** and **information gain**.
5. Builds an **optimised model** using strong features and k-fold cross-validation.
6. Compares both models and selects the best-performing one based on F1 score.
7. Applies the selected model to make predictions on unlabelled data.
8. Logs the pipeline execution time and prints key results to the console.

This function encapsulates the full modelling workflow, providing a reusable and scalable framework for supervised learning tasks. Its design supports adaptability across different datasets and ensures repeatability for experimentation and deployment.

In [79]:
def execute_ml_pipeline() -> None:
    """
    Orchestrates the entire machine learning pipeline for churn prediction.
    This function performs the following steps:
        1. Combines and loads cleaned churn data.
        2. Splits the dataset into labeled and unlabeled data.
        3. Trains a baseline XGBoost model using all features.
        4. Evaluates feature importance via correlation and information gain.
        5. Trains an optimised model using top-ranked features and cross-validation.
        6. Compares baseline and optimised models and selects the better one.
        7. Applies the best model to make predictions on unseen (unlabeled) data.
        8. Logs pipeline execution time.
    Returns:
        None. All results are printed to the console during execution.
    """

    startTime = time.time()
    print("Spark session created: ChurnXGBoost")
    print()

    randomId = uuid.uuid4()
    stringID = str(randomId)
    print(f"Executing Machine Learning Pipeline...")
    print()
    print(f"Run ID: {stringID}")
    print()

    print("***** DATA PREPARATION *****")
    print()

    print("===== Combine Cleaned Files and Load Data =====")
    churnData = concatenate_clean_churn_files()
    print()

    print("===== Cleaned Churn Metadata =====")
    get_churn_metadata(churnData)
    print()

    print("===== Split Dataframe by labels =====", end="\n")
    churnDataLabelled, churnDataUnlabelled = split_dataframe_by_label(churnData)
    churnDataLabelled = churnDataLabelled.persist(StorageLevel.MEMORY_AND_DISK) # persist labelled data frame for faster retrieval
    churnDataLabelled.count() # trigger persistence by forcing evaluation
    print()

    print("***** TRAIN AND SCORE BASE MODEL *****")
    print()

    print("===== Base Model: Separate Feature and Target Column/s =====")
    allFeatures, targetCol = separate_feature_target_cols(churnDataLabelled)
    print()

    print("===== Base Model: Correlation Analysis =====")
    strongCorrFeatures = correlation_analysis(churnDataLabelled, allFeatures, targetCol, moderateVal=0.2, strongVal=0.4)
    print()

    print("===== Base Model: Assemble Features =====")
    trainSetDf, testSetDf = feature_assemble(churnDataLabelled, allFeatures, trainSplit=0.8, testSplit=0.2)
    trainSetDf = trainSetDf.persist(StorageLevel.MEMORY_AND_DISK) # persist train data frame for faster retrieval
    testSetDf  = testSetDf.persist(StorageLevel.MEMORY_AND_DISK) # persist train data frame for faster retrieval
    trainSetDf.count(); testSetDf.count() # trigger persistence by forcing evaluation
    print()

    print("===== Base Model: Create and Score =====")
    baseModel, baselineF1Score = baseline_model(trainSetDf, testSetDf, targetCol)
    print()

    print("===== Base Model: Feature Information Gain =====")
    strongGainFeatures = feature_information_gain(baseModel, allFeatures, strongVal=30, moderateVal=10)
    print()

    print("***** TRAIN AND SCORE OPTIMISED MODEL *****")
    print()

    print("===== Optimised Model: Strongest Features =====")
    strongFeatures = get_strong_features(strongCorrFeatures, strongGainFeatures)
    print()

    print("===== Optimised Model: Create and Score =====")
    optimisedModel, optimisedF1Score = kfold_cross_validation(trainSetDf, testSetDf, strongFeatures, targetCol)
    print()

    print("***** SELECT BEST MODEL AND PREDICT *****")
    print()

    print("===== Select Best Model =====")
    bestModel, modelType, features = select_best_model(baselineF1Score, optimisedF1Score,
                                             baseModel, optimisedModel, strongFeatures, allFeatures)
    print()

    print("===== Apply Model to Predict =====")
    predict_churn_score(modelType, bestModel, churnDataUnlabelled, features)

    endTime = time.time()
    pipelineRunTime = endTime - startTime
    print(f"Machine learning pipeline complete. Run time: {timedelta(seconds=int(pipelineRunTime))}")

In [80]:
execute_ml_pipeline() # Execute pipeline

Spark session created: ChurnXGBoost

Executing Machine Learning Pipeline...

Run ID: 5a51127c-a007-4097-89e4-a5b5fc398b76

***** DATA PREPARATION *****

===== Combine Cleaned Files and Load Data =====
All available cleaned churn csv files: ['churn_clean.csv']

===== Cleaned Churn Metadata =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 25 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   age                           20000 non-null  int32  
 1   gender                        20000 non-null  int32  
 2   region_category               20000 non-null  int32  
 3   membership_category           20000 non-null  int32  
 4   joined_through_referral       20000 non-null  int32  
 5   preferred_offer_types         20000 non-null  int32  
 6   medium_of_operation           20000 non-null  int32  
 7   internet_option               20000 non-null  int32  
 8   


***** TRAIN AND SCORE OPTIMISED MODEL *****

===== Optimised Model: Strongest Features =====
Strong feature list: ['membership_category', 'avg_transaction_value', 'points_in_wallet']

===== Optimised Model: Create and Score =====
Training optimised model...
Running 3-fold CV on training data...
Tuning hyperparameters: max_depth, learning_rate, subsample...
Applying cross validation to find best F1 Score...
Optimised Model F1 Score: 0.93564
Optimised Model Precision: 0.93564
Optimised Model Recall: 0.93564
Optimised Model Accuracy: 0.93564
Optimised Model ROC AUC: 0.97807
Optimised Model Confusion Matrix:
+----------+----------------+-----+------+
|prediction|churn_risk_score|count|metric|
+----------+----------------+-----+------+
|         0|               1|  125|    FN|
|         0|               0| 1717|    TN|
|         1|               0|  126|    FP|
|         1|               1| 1932|    TP|
+----------+----------------+-----+------+


***** SELECT BEST MODEL AND PREDICT *****

# REFERENCES

### Generative AI

OpenAI. (2025). ChatGPT (June 15 Version) [Large language model]. https://chat.openai.com

    - "Provide a template using XGBoost for cross validation, hyperparamter tuning and k-folds using pyspark"

    - "Suggest some pyspark optimisation approaches"

    - "Suggest relevant scoring metrics to implement for customer churn model scoring and show python implementation"

### General

Databricks. (n.d.). Distributed training of XGBoost models using xgboost.spark. Databricks Documentation. https://docs.databricks.com/aws/en/machine-learning/train-model/xgboost-spark

DataCamp. (2023, February 22). Learn XGBoost in Python: A step-by-step tutorial. https://www.datacamp.com/tutorial/xgboost-in-python1

GeeksforGeeks. (2023, April 26). How to check the execution time of Python script? https://www.geeksforgeeks.org/python/how-to-check-the-execution-time-of-python-script/

GeeksforGeeks. (2025, May 30). Implementation of XGBoost (eXtreme Gradient Boosting). https://www.geeksforgeeks.org/machine-learning/implementation-of-xgboost-extreme-gradient-boosting/ 

GeeksforGeeks. (2025, April 28). PySpark Dataframe Split. https://www.geeksforgeeks.org/python/pyspark-dataframe-split/

Panda, M. (2019, December 24). What is the correct way to use PySpark VectorAssembler? Stack Overflow. https://stackoverflow.com/questions/59463645/what-is-the-correct-way-to-use-pyspark-vectorassembler

Roy, U. (2023, October 18). Perform xgboost prediction with pyspark dataframe. Stack Overflow. https://stackoverflow.com/questions/77320042/perform-xgboost-prediction-with-pyspark-dataframe

Stack Overflow. (2018, June 21). Convert a Spark DataFrame to Pandas DF. Stack Overflow. https://stackoverflow.com/questions/50958721/convert-a-spark-dataframe-to-pandas-df

Stack Overflow. (2015, August 29). How to change a dataframe column from String type to Double type in PySpark? Stack Overflow. https://stackoverflow.com/questions/32284620/how-to-change-a-dataframe-column-from-string-type-to-double-type-in-pyspark

XGBoost contributors. (n.d.). XGBoost parameters. XGBoost Documentation. https://xgboost.readthedocs.io/en/stable/parameter.html

XGBoost Contributors. (n.d.). Distributed XGBoost with PySpark: SparkXGBClassifier. XGBoost Documentation. https://xgboost.readthedocs.io/en/latest/tutorials/spark_estimator.html#sparkxgbclassifier


